# Training an Object Segmentation Workshop
### By: Aaron Gabrielle C. Dichoso
### From: DLSU - Center of Imaging and Visual Innovations (CIVI)
May 27, 2025

## 1. Importing Libraries

In [ ]:
import torch
from torchvision.models.segmentation import fcn_resnet50 #Pytorch built-in model
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader
import os
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

from tqdm import tqdm

In [ ]:
COCO_PATH = './dataset/'
TRAIN_IMG_DIR = os.path.join(COCO_PATH, 'coco_sample/train')
TEST_IMG_DIR = os.path.join(COCO_PATH, 'coco_sample/test')
VAL_IMG_DIR = os.path.join(COCO_PATH, 'coco_sample/val')
TRAIN_ANN_FILE = os.path.join(COCO_PATH, 'coco_sample/train.json')
TEST_ANN_FILE = os.path.join(COCO_PATH, 'coco_sample/test.json')
VAL_ANN_FILE = os.path.join(COCO_PATH, 'coco_sample/val.json')

## 2. Notebook configurations
Batch Size: The Number of Images passed to the model during training in one forward pass

Classes: Subset of 12 objects + 1 background class from the standard MSCOCO classes

["dog", "cat", "person", "chair", "mouse", "remote", "keyboard", "cell phone", "cup", "fork", "knife", "spoon"]

Epochs: Number of Iterations that the training images will be passed to the model.

Learning Rate: Affects the strength of adjustments applied to the model during training.

Device: Use CUDA if available, else use the CPU

In [ ]:
BATCH_SIZE = 4
NUM_CLASSES = 12 + 1
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 3. Dataset Loader

A dataset loader is utilized in this notebook to allow modifications to the original MSCOCO dataset:

1. Instead of the 91 classes in MSCOCO, only 13 classes are used
2. Allows you to apply transformations to images before loading the image

In [ ]:
class CocoSegmentation(torch.utils.data.Dataset):
    def __init__(self, root, annFile):
        self.root = root
        self.coco = COCO(annFile)
        self.ids = list(self.coco.imgs.keys())
        self.cat_ids = self.coco.getCatIds()
        print(self.cat_ids)

        self.target_size = (480, 640)  # (height, width)

        #Remap indexes from original labels to new labels
        #I.E. ids = {45, 23, 54} -> {1, 2, 3}
        self.cat_id_to_index = {cat_id: idx + 1 for idx, cat_id in enumerate(self.cat_ids)} 

        #Class names of labels
        self.class_names = {
            self.cat_id_to_index[c['id']]: c['name']
            for c in self.coco.loadCats(self.cat_ids)
        }
        
        #Default class for unidentified objects
        self.class_names[0] = "background"
        
        print(self.class_names)

    #Used by data loader to access images and masks
    def __getitem__(self, idx):
        #Annotated labels of objects
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id) 
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]

        # Load and resize image
        path = img_info['file_name']
        img_path = os.path.join(self.root, path)
        img = Image.open(img_path).convert('RGB')
        img = F.resize(img, self.target_size)
        img_tensor = F.to_tensor(img) #Convert to Tensor to be used in CNN

        # Create and resize semantic mask
        height, width = img_info['height'], img_info['width']
        mask = np.zeros((height, width), dtype=np.uint8)

        #Remap original MSCOCO Labels for segmentation masks to the new labels used in this dataset
        for ann in anns:
            if 'segmentation' in ann and isinstance(ann['segmentation'], list):
                rles = coco_mask.frPyObjects(ann['segmentation'], height, width) #Convert polygon segmentation to RLE - Run-Length Encoding
                rle = coco_mask.merge(rles) #Combine into one RLE
                m = coco_mask.decode(rle) #Decode RLE to binary mask
                class_idx = self.cat_id_to_index.get(ann['category_id'], 0) #Map original category ID to new class index, default to 0 (background) if not found
                mask[m == 1] = class_idx

        # Resize mask to match resized image
        mask = Image.fromarray(mask)
        mask = F.resize(mask, self.target_size, interpolation=Image.NEAREST)
        mask_tensor = torch.as_tensor(np.array(mask), dtype=torch.long) #Convert Mask to Tensor

        return img_tensor, mask_tensor

    def __len__(self):
        return len(self.ids)

## 4. Load Datasets

In [ ]:
train_dataset = CocoSegmentation(TRAIN_IMG_DIR, TRAIN_ANN_FILE)
test_dataset = CocoSegmentation(TEST_IMG_DIR, TEST_ANN_FILE)
val_dataset = CocoSegmentation(VAL_IMG_DIR, VAL_ANN_FILE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

## 5. FCN Model Configurations

Get the FCN model from pytorch, and modify it to use the custom number of classes

In [ ]:
model = fcn_resnet50(pretrained=False, num_classes=NUM_CLASSES)
model.to(DEVICE) #Uses CUDA if available
print(model)

Criterion: Measure of Model Performance used during training.

Optimizer: Method used to determine the optimal weights during training.

Scheduler: Decays the Learning Rate over epochs

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 6. Training Loop

General Flow is as follows:
1. Set the model into training mode
2. For each epoch, do the following:

    a. Load the Images and Masks to the Device

    b. Zero out existing gradients

    c. Perform a Forward Pass

    d. Compute the Prediction Performance / Loss

    e. Perform a backward pass and update weights (Backpropagation)

    f. save the model checkpoint
    

In [ ]:
def train(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device):
    if not os.path.exists("checkpoints"):
        os.makedirs("checkpoints")  # Create a directory to save checkpoints

    best_val_loss = float('inf')

    for epoch in range(num_epochs):  # For each epoch
        model.train()
        total_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False) #Use tqdm for tracking training progress

        for images, masks in progress_bar:
            images, masks = images.to(device), masks.to(device)  # Load Images and Masks to Device

            optimizer.zero_grad()  # Zero existing Gradients Just in Case
            outputs = model(images)['out']  # Forward Pass
            loss = criterion(outputs, masks)  # Compute Prediction Loss
            loss.backward()  # Backward Pass and Updating Weights
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.4f}")

        # Validation step
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Validation", leave=False)
            for val_images, val_masks in val_bar:
                val_images, val_masks = val_images.to(device), val_masks.to(device)
                val_outputs = model(val_images)['out']
                loss_val = criterion(val_outputs, val_masks)
                val_loss += loss_val.item()
                val_bar.set_postfix(val_loss=loss_val.item())

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}")

        # Step the scheduler
        scheduler.step()

        # Save model checkpoint
        checkpoint_path = f"checkpoints/checkpoint_epoch_{epoch+1}.pth"
        torch.save(model.state_dict(), checkpoint_path)

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_path = "checkpoints/best_model.pth"
            torch.save(model.state_dict(), best_path)
            print(f"Best model saved with val loss: {best_val_loss:.4f}")


In [ ]:
train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
)

## 7. Evaluation Metrics

#### Pixel Accuracy: Used to examine overall accuracy.
Total amount of pixels correctly assigned with their labels / All pixels


#### Mean Intersection over Union (IoU): Used to examine the the performance of the model, with structure in mind.

Take the Intersection of the prediction to the ground truth of the image. Divide by its Union. Average across all images.


In [ ]:
def pixel_accuracy(preds, labels):
    valid = labels >= 0
    correct = (preds == labels) & valid
    return correct.sum().item() / valid.sum().item()

def mean_iou(preds, labels, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_inds = (preds == cls)
        label_inds = (labels == cls)
        intersection = (pred_inds & label_inds).sum().item()
        union = (pred_inds | label_inds).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return sum(ious) / len(ious) if ious else 0.0

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    accs, ious = [], []
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)['out']
            preds = torch.argmax(outputs, dim=1)

            accs.append(pixel_accuracy(preds.cpu(), masks.cpu()))
            ious.append(mean_iou(preds.cpu(), masks.cpu(), NUM_CLASSES))

    print(f"Pixel Accuracy: {np.mean(accs):.4f}")
    print(f"Mean IoU: {np.mean(ious):.4f}")

In [ ]:
def load_model(path):
    model = fcn_resnet50(weights=None, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

In [ ]:
MODEL_PATH = "checkpoints/best_model.pth"
saved_model = load_model(MODEL_PATH)

In [ ]:
evaluate(saved_model, test_loader)

## 8. Visualization

Use images from the test dataset to demonstrate model performance across classes to find possible rooms of improvement.

From seeing the predictions. The model predicts people well, but not sparse entries like spoons.

Indicates that it needs more examples to learn from for those classes (Limitation due to sampled dataset).


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as FNN

def visualize_random_samples(model, dataset, class_names, num_samples=5):
    model.eval()
    indices = random.sample(range(len(dataset)), num_samples)

    # Build sorted list of labels
    label_list = sorted(class_names.keys())
    num_labels = max(label_list) + 1

    # Use a vivid colormap
    base_cmap = plt.cm.get_cmap('nipy_spectral', num_labels)
    label_colors = {label: base_cmap(label / num_labels) for label in label_list}

    def colorize_mask(mask_array):
        rgb = np.zeros((*mask_array.shape, 3), dtype=np.float32)
        for label in np.unique(mask_array):
            rgb[mask_array == label] = label_colors.get(label, (0, 0, 0))[:3]
        return rgb

    # Create figure with 3 columns: Image | Ground Truth | Prediction
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)  # make sure it's always 2D

    for row_idx, idx in enumerate(indices):
        image, mask = dataset[idx]
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(DEVICE))['out']
            probs = FNN.softmax(output, dim=1)[0]
            pred = torch.argmax(probs, dim=0).cpu().numpy()

            # Compute confidences
            unique_labels = np.unique(pred)
            confs = {label: probs[label][pred == label].mean().item() for label in unique_labels}

        pred_rgb = colorize_mask(pred)
        mask_rgb = colorize_mask(mask.numpy())

        # Image
        axes[row_idx, 0].imshow(image.permute(1, 2, 0))
        axes[row_idx, 0].set_title(f"Sample {idx} - Image")
        axes[row_idx, 0].axis('off')

        # Ground Truth
        axes[row_idx, 1].imshow(mask_rgb)
        axes[row_idx, 1].set_title("Ground Truth")
        axes[row_idx, 1].axis('off')

        # Prediction
        axes[row_idx, 2].imshow(pred_rgb)
        axes[row_idx, 2].set_title("Prediction")
        axes[row_idx, 2].axis('off')

        # Optional: print to console
        print(f"Sample index: {idx}")
        print("Class labels in prediction and average confidence:")
        for label in unique_labels:
            name = class_names.get(label, f"Unknown (ID {label})")
            confidence = confs[label]
            print(f"Label {label}: {name} - Confidence: {confidence:.3f}")
        print("-" * 30)

    # Add a legend once (bottom center of the plot)
    legend_patches = [
        mpatches.Patch(color=label_colors[l][:3], label=f"{class_names.get(l, 'Unknown')} ({l})")
        for l in label_list
    ]
    fig.legend(
        handles=legend_patches,
        loc='upper center',
        ncol=min(len(legend_patches), 6),
        bbox_to_anchor=(0.5, 1.02),
        fontsize='small'
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])  # leave space at bottom for legend
    plt.show()

In [ ]:
visualize_random_samples(saved_model, test_dataset, test_dataset.class_names, 5)

# 9. Instance Segmentation with Mask R-CNN
This section extends the semantic segmentation workflow to **instance segmentation** using Mask R-CNN. Unlike FCN, Mask R-CNN predicts a separate mask, class label, and bounding box for each object instance.

In [ ]:
BATCH_SIZE = 4
NUM_CLASSES = 12 + 1
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 9.1 Dataset Wrapper for Mask R-CNN

In [ ]:
from torchvision.transforms import functional as F
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

import torch
import numpy as np
import os
from PIL import Image

class CocoInstanceSegmentation(torch.utils.data.Dataset):
    def __init__(self, root, annFile):
        self.root = root
        self.coco = COCO(annFile)

        self.ids = list(sorted(self.coco.imgs.keys()))
        self.cat_ids = list(sorted(self.coco.getCatIds()))

        self.target_size = (480, 640)  # (height, width)

        # Remap original COCO IDs -> contiguous IDs
        # Example: {45,23,54} -> {1,2,3}
        self.cat_id_to_index = {
            cat_id: idx + 1
            for idx, cat_id in enumerate(self.cat_ids)
        }

        self.class_names = {}
        for cat_id in self.cat_ids:
            cat_list = self.coco.loadCats([cat_id])
            if cat_list:
                self.class_names[self.cat_id_to_index[cat_id]] = cat_list[0]["name"]
            else:
                self.class_names[self.cat_id_to_index[cat_id]] = f"unknown_{cat_id}"

        self.class_names[0] = "background"

        print("Original category IDs:")
        print(self.cat_ids)

        print("\nRemapped classes:")
        print(self.class_names)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=image_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(image_id)[0]

        img_path = os.path.join(self.root, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")

        # Keep track of original dimensions for scaling targets
        orig_w, orig_h = image.size

        # Scale factors for bounding boxes
        w_ratio = self.target_size[1] / orig_w
        h_ratio = self.target_size[0] / orig_h

        image = F.resize(image, self.target_size)
        image_tensor = F.to_tensor(image)

        boxes = []
        labels = []
        masks = []
        areas = []
        iscrowd = []

        for ann in anns:
            # Skip malformed annotations
            if ann["bbox"][2] <= 0 or ann["bbox"][3] <= 0:
                continue

            x, y, w, h = ann["bbox"]
            x1 = x * w_ratio
            y1 = y * h_ratio
            x2 = (x + w) * w_ratio
            y2 = (y + h) * h_ratio
            boxes.append([x1, y1, x2, y2])

            #Remap category IDs just like semantic segmentation dataset
            #Skip annotations whose category_id is not in our map (can happen when the JSON has stray IDs not listed in the categories section)
            if ann["category_id"] not in self.cat_id_to_index:
                boxes.pop()  # undo the box we already appended
                continue
            labels.append(self.cat_id_to_index[ann["category_id"]])

            mask = self.coco.annToMask(ann)
            mask_tensor = torch.as_tensor(mask, dtype=torch.uint8).unsqueeze(0) # [1, H_orig, W_orig]
            mask_resized = F.resize(mask_tensor, self.target_size, interpolation=F.InterpolationMode.NEAREST)
            masks.append(mask_resized.squeeze(0))
            areas.append(ann["area"])
            iscrowd.append(ann.get("iscrowd", 0))

        # Handle images with no objects
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            masks = torch.zeros((0, self.target_size[0], self.target_size[1]), dtype=torch.uint8)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            #Object exists, convert model outputs to tensors
            boxes = torch.as_tensor(boxes, dtype=torch.float32) #Bounding boxes as Mask R-CNN does detection as well as segmentation
            labels = torch.as_tensor(labels, dtype=torch.int64) #bounding box labels
            masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8) #Instance maks
            areas = torch.as_tensor(areas, dtype=torch.float32) #Bounding box areas
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64) #Boolean to check if the annotation is a crowd of objects

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([image_id]),
            "area": areas,
            "iscrowd": iscrowd
        }

        return image_tensor, target

    def __len__(self):
        return len(self.ids)


def collate_fn(batch):
    return tuple(zip(*batch))

## 9.2 Load Instance Segmentation Dataset

In [ ]:
instance_train_dataset = CocoInstanceSegmentation(TRAIN_IMG_DIR, TRAIN_ANN_FILE)
instance_val_dataset = CocoInstanceSegmentation(VAL_IMG_DIR, VAL_ANN_FILE)
instance_test_dataset = CocoInstanceSegmentation(TEST_IMG_DIR, TEST_ANN_FILE)

instance_train_loader = DataLoader(
    instance_train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

instance_val_loader = DataLoader(
    instance_val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

instance_test_loader = DataLoader(
    instance_test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)


## 9.3 Mask R-CNN Model Configuration

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

maskrcnn_model = maskrcnn_resnet50_fpn(weights=None)

in_features = maskrcnn_model.roi_heads.box_predictor.cls_score.in_features
maskrcnn_model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

in_features_mask = maskrcnn_model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256

maskrcnn_model.roi_heads.mask_predictor = MaskRCNNPredictor(
    in_features_mask,
    hidden_layer,
    NUM_CLASSES
)

maskrcnn_model.to(DEVICE)


## 9.4 Training Loop

In [ ]:
mask_optimizer = torch.optim.Adam(
    maskrcnn_model.parameters(),
    lr=LEARNING_RATE
)

def train_maskrcnn(model, dataloader, optimizer, epochs=10):
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0

        #Training Step
        for images, targets in tqdm(dataloader):
            images = [img.to(DEVICE) for img in images]
            targets = [
                {k: v.to(DEVICE) for k, v in t.items()}
                for t in targets
            ]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()

        #Validation step
        val_loss_list = []
        model.eval()
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} Validation", leave=False)
            for val_images, val_targets in val_bar:
                val_images = [img.to(DEVICE) for img in val_images]

                # Mask R-CNN at eval time returns predictions, not losses
                # To get a loss signal, we temporarily switch back to train mode
                model.train()
                loss_dict = model(val_images, [{k: v.to(DEVICE) for k, v in t.items()} for t in val_targets])
                loss_val = sum(loss_dict.values())
                val_loss_list.append(loss_val.item())
                val_bar.set_postfix(val_loss=loss_val.item())
                model.eval()

        val_loss = np.mean(val_loss_list)

        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(dataloader):.4f} | Validation Loss: {val_loss:.4f}")


In [ ]:
train_maskrcnn(
    maskrcnn_model,
    instance_train_loader,
    mask_optimizer,
    epochs=NUM_EPOCHS
)

torch.save(
    maskrcnn_model.state_dict(),
    "checkpoints/maskrcnn_best_model.pth"
)


## 9.5 Evaluation (Bounding Box and Mask Metrics)
For rigorous evaluation, use the COCO evaluator which reports AP, AP50, AP75 and mask AP metrics.

In [44]:
def pixel_accuracy(preds, labels):
    valid = labels >= 0
    correct = (preds == labels) & valid
    return correct.sum().item() / valid.sum().item()

def mean_iou(preds, labels, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_inds = (preds == cls)
        label_inds = (labels == cls)
        intersection = (pred_inds & label_inds).sum().item()
        union = (pred_inds | label_inds).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return sum(ious) / len(ious) if ious else 0.0

In [45]:
from torchvision.datasets import CocoDetection
from torchvision.models.detection import MaskRCNN
from torchvision.ops import box_iou

def evaluate_maskrcnn(model, dataloader, score_threshold=0.5):
    model.eval()

    accs, ious = [], []

    with torch.no_grad():
        for images, targets in tqdm(dataloader):
            images = [img.to(DEVICE) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                H, W = target["masks"].shape[-2:]

                #output["masks"] is [N, 1, H, W] float, one mask per instance
                #output["labels"] is [N] with the class index per instance
                pred_semantic = torch.zeros((H, W), dtype=torch.int64)
                for mask, score, label in zip(
                    output["masks"].cpu(),
                    output["scores"].cpu(),
                    output["labels"].cpu()
                ):
                    if score < score_threshold:
                        continue
                    binary_mask = mask[0] > 0.5          # [H, W] bool
                    pred_semantic[binary_mask] = label    # Collapse into semantic mask

                # Compare with ground truth
                # target["masks"] is [N, H, W] uint8 binary, target["labels"] is [N]
                gt_semantic = torch.zeros((H, W), dtype=torch.int64)
                for mask, label in zip(target["masks"], target["labels"]):
                    gt_semantic[mask.bool()] = label      # Collapse into semantic mask

                accs.append(pixel_accuracy(pred_semantic, gt_semantic))
                ious.append(mean_iou(pred_semantic, gt_semantic, NUM_CLASSES))

    print(f"Pixel Accuracy: {np.mean(accs):.4f}")
    print(f"Mean IoU:       {np.mean(ious):.4f}")

In [46]:
evaluate_maskrcnn(
    maskrcnn_model,
    instance_test_loader
)


100%|██████████| 45/45 [01:44<00:00,  2.31s/it]

Pixel Accuracy: 0.8676
Mean IoU:       0.3831


## 9.6 Visualization of Predicted Instances

In [47]:
def visualize_maskrcnn_predictions(model, dataset, num_samples=3, score_threshold=0.5):
    model.eval()

    for _ in range(num_samples):
        idx = random.randint(0, len(dataset)-1)

        image, target = dataset[idx]

        with torch.no_grad():
            prediction = model([image.to(DEVICE)])[0]

        img_np = image.permute(1,2,0).cpu().numpy()

        fig, ax = plt.subplots(figsize=(8,8))
        ax.imshow(img_np)

        for instance_id, (box, score, label, mask) in enumerate(zip(
            prediction["boxes"].cpu(),
            prediction["scores"].cpu(),
            prediction["labels"].cpu(),
            prediction["masks"].cpu()
        )):
            if score < score_threshold:
                continue

            x1, y1, x2, y2 = box.numpy()

            rect = patches.Rectangle(
                (x1, y1),
                x2-x1,
                y2-y1,
                fill=False
            )
            ax.add_patch(rect)

            class_name = dataset.class_names.get(label.item(), str(label.item()))
            ax.text(
                x1, y1 - 4,
                f"[{instance_id}] {class_name}: {score:.2f}",  # <-- id added
                color="white",
                fontsize=8,
                bbox=dict(facecolor="black", alpha=0.5, pad=2, edgecolor="none")
            )

            binary_mask = mask[0].numpy() > 0.5
            ax.imshow(
                np.ma.masked_where(~binary_mask, binary_mask),
                alpha=0.4
            )

        plt.axis("off")
        plt.show()

In [48]:
visualize_maskrcnn_predictions(
    maskrcnn_model,
    instance_test_dataset,
    num_samples=5
)


NameError: name 'random' is not defined